# Fuzzy matching: rapikan kategori kotor, cocokkan ke acuan, cari teks hampir kembar
**Kapan dipakai:** kolom kategori/teks pendek yang ditulis beda-beda ("Jakarta", "jakrta",
"JAKARTA "), teks yang harus dipetakan ke daftar resmi (judul produk → kategori), atau baris
test yang hampir sama dengan baris train (salin label / jadikan fitur). Pustaka: `rapidfuzz`.
Dua tabel tanpa kunci bersama → `contoh/multitabel/entity_resolution` (model pasangan).

1. **Rapikan kategori**: nilai mirip (skor ≥ `CLUSTER_SCORE`) digabung ke nilai yang paling
   sering muncul. Peta ditulis ke CSV untuk dicek mata.
2. **Cocokkan ke acuan**: tiap teks `LOOKUP_COL` dicari padanan terbaiknya di `REFERENCE`.
3. **Hampir kembar train↔test**: kandidat dari TF-IDF huruf, skor akhir dari rapidfuzz.

Hasil: `train_fuzzy.csv` / `test_fuzzy.csv` = data asli + kolom baru. Arahkan `TRAIN_CSV` /
`TEST_CSV` notebook lain (`stacking_teks_tabular`, `teks_tabular`) ke file ini.

In [ ]:
import os

import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process, utils
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

## Konfigurasi
Skor rapidfuzz 0–100. `fuzz.ratio` = kemiripan huruf per huruf (aman untuk salah ketik),
`fuzz.partial_ratio` = acuan muncul di dalam teks yang lebih panjang, `fuzz.token_sort_ratio` =
urutan kata diabaikan, `fuzz.WRatio` = gabungan otomatis (paling longgar, rawan salah gabung).

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/fuzzy_kategori"
DATA_FOLDER = "../../../data_latihan/text/teks_tabular"      # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/baju_wanita_train.csv"           # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/baju_wanita_test.csv"             # GANTI saat lomba
KEY_CSV = f"{DATA_FOLDER}/baju_wanita_test_labels.csv"       # GANTI saat lomba (kunci, kalau ada)

ID_COL = "id"                          # GANTI saat lomba
TARGET = "rating"                      # GANTI saat lomba
MESSY_COLS = ["class_name"]            # GANTI saat lomba: kolom kategori yang ditulis beda-beda
LOOKUP_COL = "title"                   # GANTI saat lomba: teks yang dicocokkan ke acuan
REFERENCE = None                       # GANTI: daftar resmi, mis. ["Jakarta", "Bandung"]
TEXT_COLS = ["title", "review_text"]   # GANTI saat lomba: untuk cari hampir kembar

CLUSTER_SCORER = fuzz.ratio
# CLUSTER_SCORER = fuzz.token_sort_ratio   # alt: urutan kata beda ("Bank BCA" vs "BCA Bank")
LOOKUP_SCORER = fuzz.partial_ratio
# LOOKUP_SCORER = fuzz.WRatio              # alt: lebih longgar
CLUSTER_SCORE = 85       # naikkan kalau nilai yang beda arti ikut tergabung
LOOKUP_SCORE = 90       # 80 terlalu longgar untuk acuan pendek ("not red" → "Trend")
TWIN_SCORE = 95          # skor teks dianggap hampir kembar
N_CANDIDATES = 5
SEED = 42

## Baca data

In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
for col in TEXT_COLS + [LOOKUP_COL]:
    train[col] = train[col].fillna("").astype(str)
    test[col] = test[col].fillna("").astype(str)
for col in MESSY_COLS:
    train[col] = train[col].fillna("kosong").astype(str)
    test[col] = test[col].fillna("kosong").astype(str)
print("train:", train.shape, "| test:", test.shape)

## Kotori kategori demo (HAPUS saat lomba)
Data demo sudah rapi, jadi 15% nilai `MESSY_COLS` dirusak: satu huruf hilang, huruf kecil
semua, huruf besar semua, spasi lebih. Saat lomba datanya memang sudah kotor.

In [ ]:
rng = np.random.default_rng(SEED)
clean_train = train[MESSY_COLS].copy()


def make_messy(value):
    choice = rng.integers(4)
    if choice == 0 and len(value) > 4:
        position = rng.integers(1, len(value) - 1)
        return value[:position] + value[position + 1:]
    if choice == 1:
        return value.lower()
    if choice == 2:
        return value.upper() + " "
    return " " + value


for table in [train, test]:
    for col in MESSY_COLS:
        values = table[col].to_numpy(dtype=object)
        for row in np.flatnonzero(rng.random(len(values)) < 0.15):
            values[row] = make_messy(values[row])
        table[col] = values
print(train[MESSY_COLS[0]].value_counts().head(15))

## 1. Rapikan kategori
Nilai diurutkan dari yang paling sering. Nilai yang belum punya kelompok jadi "nilai baku", lalu
semua nilai lain yang skornya ≥ `CLUSTER_SCORE` ke nilai baku itu masuk kelompoknya.
`utils.default_process` = huruf kecil, buang tanda baca dan spasi tepi sebelum dibandingkan.

In [ ]:
def build_canonical(values):
    counts = pd.Series(values).value_counts()
    names = list(counts.index)
    scores = process.cdist(names, names, scorer=CLUSTER_SCORER, processor=utils.default_process,
                           workers=-1)
    canonical = {}
    rows = []
    for i, name in enumerate(names):
        if name in canonical:
            continue
        canonical[name] = name
        for other in np.flatnonzero(scores[i] >= CLUSTER_SCORE):
            if names[other] not in canonical:
                canonical[names[other]] = name
                rows.append({"nilai": names[other], "jumlah": counts[names[other]],
                             "jadi": name, "skor": round(float(scores[i, other]), 1)})
    return canonical, pd.DataFrame(rows, columns=["nilai", "jumlah", "jadi", "skor"])


print("uji:", build_canonical(["Dress", "Dress", "dress ", "Pants"])[0])

In [ ]:
merge_tables = []
for col in MESSY_COLS:
    values = pd.concat([train[col], test[col]])
    canonical, merged = build_canonical(values)
    train[f"{col}_rapi"] = train[col].map(canonical)
    test[f"{col}_rapi"] = test[col].map(canonical)
    merged["kolom"] = col
    merge_tables.append(merged)
    print(col, "| nilai unik:", values.nunique(), "→", len(set(canonical.values())))
merge_table = pd.concat(merge_tables, ignore_index=True)
merge_table.to_csv(f"{OUTPUT_FOLDER}/peta_kategori.csv", index=False)
lowest = merge_table.sort_values("skor")
print(lowest.head(15).to_string(index=False))

Cek baris berskor terendah dulu: paling mungkin salah gabung ("Pria" ↔ "Prima"). Kalau ada,
naikkan `CLUSTER_SCORE` atau koreksi `peta_kategori.csv` lalu petakan ulang.

**Cek demo (HAPUS saat lomba):** berapa persen nilai kembali ke nilai aslinya.

In [ ]:
for col in MESSY_COLS:
    restored = (train[f"{col}_rapi"] == clean_train[col]).mean()
    print(col, "| kembali ke nilai asli:", round(restored, 4))

## 2. Cocokkan teks ke daftar acuan
`REFERENCE = None` → acuan = nilai baku kolom pertama `MESSY_COLS` (demo: judul "Cute dress" →
"Dresses"). Skor di bawah `LOOKUP_SCORE` → "tidak_ada". Kolom skor ikut jadi fitur.

In [ ]:
reference = REFERENCE
if reference is None:
    reference = sorted(set(train[f"{MESSY_COLS[0]}_rapi"]) - {"kosong"})
reference = np.array(reference, dtype=object)


def lookup(texts):
    scores = process.cdist(list(texts), list(reference), scorer=LOOKUP_SCORER,
                           processor=utils.default_process, workers=-1)
    best_score = scores.max(axis=1)
    match = np.where(best_score >= LOOKUP_SCORE, reference[scores.argmax(axis=1)], "tidak_ada")
    return match, best_score


for table in [train, test]:
    match, best_score = lookup(table[LOOKUP_COL])
    table[f"{LOOKUP_COL}_acuan"] = match
    table[f"{LOOKUP_COL}_acuan_skor"] = best_score
found = train[train[f"{LOOKUP_COL}_acuan"] != "tidak_ada"]
print("acuan:", len(reference), "| baris train dapat padanan:", round(len(found) / len(train), 3))
agree = (found[f"{LOOKUP_COL}_acuan"] == found[f"{MESSY_COLS[0]}_rapi"]).mean()
print("padanan = kategori baris itu sendiri:", round(agree, 3))
borderline = found.sort_values(f"{LOOKUP_COL}_acuan_skor")
columns = [LOOKUP_COL, f"{LOOKUP_COL}_acuan", f"{LOOKUP_COL}_acuan_skor"]
borderline[columns].head(200).to_csv(f"{OUTPUT_FOLDER}/cocok_acuan_batas.csv", index=False)
print(borderline[columns].head(10).to_string(index=False))

## 3. Teks hampir kembar train↔test
Tetangga terdekat TF-IDF huruf (cepat) → dicek ulang dengan `fuzz.ratio` (teliti). Untuk train,
tetangga dicari di train lain (dirinya dibuang), jadi kolom `kembar_*` adil dipakai sebagai fitur.

In [ ]:
def joined_text(table):
    return table[TEXT_COLS].agg(" ".join, axis=1)


text_train = joined_text(train)
text_test = joined_text(test)
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True,
                             max_features=300000)
vectorizer.fit(pd.concat([text_train, text_test]))
matrix_train = vectorizer.transform(text_train)
matrix_test = vectorizer.transform(text_test)
finder = NearestNeighbors(n_neighbors=N_CANDIDATES + 1, metric="cosine").fit(matrix_train)
_, index_train = finder.kneighbors(matrix_train)
_, index_test = finder.kneighbors(matrix_test, n_neighbors=N_CANDIDATES)
print("TF-IDF huruf:", matrix_train.shape)

In [ ]:
def best_twin(texts, candidates, ref_texts, own_rows):
    best_row = np.zeros(len(texts), dtype=int)
    best_score = np.zeros(len(texts))
    for i, text in enumerate(texts):
        for candidate in candidates[i]:
            if candidate == own_rows[i]:
                continue
            score = fuzz.ratio(text, ref_texts[candidate], processor=utils.default_process)
            if score > best_score[i]:
                best_score[i] = score
                best_row[i] = candidate
    return best_row, best_score


train_texts = text_train.to_numpy()
twin_train, score_train = best_twin(train_texts, index_train, train_texts, np.arange(len(train)))
twin_test, score_test = best_twin(text_test.to_numpy(), index_test, train_texts,
                                  np.full(len(test), -1))
labels = train[TARGET].to_numpy()
train["kembar_skor"] = score_train
train["kembar_label"] = labels[twin_train]
test["kembar_skor"] = score_test
test["kembar_label"] = labels[twin_test]
close = score_train >= TWIN_SCORE
same = (labels[twin_train] == labels)[close].mean()
print(f"train punya kembar ≥ {TWIN_SCORE}: {close.sum()} | labelnya sama: {same:.3f}")
print(f"test punya kembar di train ≥ {TWIN_SCORE}: {(score_test >= TWIN_SCORE).sum()}")

**Salin label?** Hanya kalau "labelnya sama" di train tinggi (≥ 0,9). Caranya: ganti tebakan
test yang `kembar_skor ≥ TWIN_SCORE` dengan `kembar_label` (lihat `pasca_prediksi`). Kalau
rendah, kembar beda label = label berisik → lihat `label_berisik`.

In [ ]:
twins = pd.DataFrame()
twins[ID_COL] = test[ID_COL]
twins["id_train"] = train[ID_COL].to_numpy()[twin_test]
twins["skor"] = score_test
twins["label_train"] = labels[twin_test]
twins["teks_test"] = text_test.str.slice(0, 150)
twins["teks_train"] = text_train.str.slice(0, 150).to_numpy()[twin_test]
twins = twins.sort_values("skor", ascending=False)
twins.to_csv(f"{OUTPUT_FOLDER}/kembar_train_test.csv", index=False)
score_bins = pd.cut(score_test, [0, 80, 90, 95, 99, 100.1], right=False)
print(pd.Series(score_bins).value_counts(sort=False))

## Simpan data + kolom baru

In [ ]:
new_cols = []
for col in train.columns:
    if col.endswith("_rapi") or col.startswith(f"{LOOKUP_COL}_acuan") or col.startswith("kembar_"):
        new_cols.append(col)
train.to_csv(f"{OUTPUT_FOLDER}/train_fuzzy.csv", index=False)
test.to_csv(f"{OUTPUT_FOLDER}/test_fuzzy.csv", index=False)
print("kolom baru:", new_cols)
print("ditulis:", f"{OUTPUT_FOLDER}/train_fuzzy.csv", "+ test_fuzzy.csv")

## Cek salin label dengan kunci jawaban (HAPUS saat lomba)

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].values, index=key[ID_COL].values)
truth = test[ID_COL].map(key_by_id).to_numpy()
for limit in [90, 95, 99]:
    rows = score_test >= limit
    print(f"kembar ≥ {limit}: {rows.sum()} baris | label train = label test:",
          round((test["kembar_label"].to_numpy()[rows] == truth[rows]).mean(), 3))